# KVO_05 — SAM 3 zero-shot Baego crown segmentation benchmark

**Question:** can official Meta SAM 3 reproduce high-quality manually digitized tropical tree crowns from the Baego UAV RGB orthomosaic, without task-specific training?

This clean revision deliberately starts with **Baego only** using `Annotations/Crown_Nadine.shp`. The polygons do **not** need to form a complete wall-to-wall crown inventory: this notebook is a **prompt-conditioned delineation benchmark**, so each digitized crown is an independent reference object. Incomplete regional coverage is therefore fine as long as the digitized crowns themselves are high-quality and reasonably representative.

Workflow:
1. host geospatial environment prepares Baego RGB chips and reference masks;
2. official SAM3 runs in an isolated environment;
3. each manual crown bbox is expanded slightly and supplied as a geometric prompt;
4. SAM3 mask is compared with the manual crown using IoU, Dice, area bias and centroid displacement.

This is not yet automatic crown discovery and does not use Musanga class labels.


## 0 — Host environment and Drive

Use a **GPU runtime**. Do not install SAM3 into the Colab host Python. This notebook intentionally keeps the geospatial and SAM3 dependency stacks separate.


In [ ]:
import sys, subprocess, os, json, random, warnings, gc, shutil
from pathlib import Path
print("Host Python:",sys.version)
subprocess.run(["nvidia-smi"],check=False)
# Install/repair only the host geospatial stack. SAM3 is installed later in /content/sam3env.
!pip -q install -U "numpy>=2,<2.6" "huggingface_hub<2" geopandas rasterio pyogrio shapely scikit-learn matplotlib tqdm pillow


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
from tqdm.auto import tqdm
from PIL import Image
from google.colab import drive
warnings.filterwarnings("ignore")

SEED=42
random.seed(SEED); np.random.seed(SEED)
DRIVE_ROOT=Path("/content/drive"); MYDRIVE=DRIVE_ROOT/"MyDrive"
def healthy():
    try: return MYDRIVE.exists() and any(MYDRIVE.iterdir())
    except OSError: return False
if not healthy(): drive.mount(str(DRIVE_ROOT),force_remount=True)

ROOT=MYDRIVE/"Colab Notebooks"/"Musanga"; GEO=ROOT/"data"/"Geo_Ref"; AD=GEO/"Annotations"
OUT=ROOT/"KVO_work"/"KVO_05_SAM3_Baego_Crown_Nadine"; OUT.mkdir(parents=True,exist_ok=True)
CHIPS=OUT/"benchmark_chips"; CHIPS.mkdir(exist_ok=True)
SAMOUT=OUT/"sam3_masks"; SAMOUT.mkdir(exist_ok=True)
VIS=OUT/"examples"; VIS.mkdir(exist_ok=True)

SITE="Baego"
RASTER=GEO/"Baego_georef.tif"
ANNOT=AD/"Crown_Nadine.shp"
assert RASTER.exists(), RASTER
assert ANNOT.exists(), ANNOT

MIN_AREA_M2=5.0
N_CROWNS=200       # set None to use all eligible Crown_Nadine polygons
CROP_MARGIN_FRAC=.75
BOX_EXPAND_FRAC=.15
MIN_CROP_PX=384
FORCE_PREP=False
FORCE_SAM=False
print("Host numpy",np.__version__,"rasterio",rasterio.__version__,"geopandas",gpd.__version__)
print("Raster:",RASTER)
print("Reference crowns:",ANNOT)
print("OUT:",OUT)


## 1 — Prepare the Baego Crown_Nadine benchmark

All valid polygons in `Crown_Nadine.shp` with area ≥5 m² are eligible. To keep the first benchmark compact, at most 200 crowns are sampled reproducibly across crown-area quartiles. Set `N_CROWNS=None` later if you want to run every eligible crown.

The manual polygons may cover only selected parts of Baego. That is acceptable here because SAM3 is evaluated only on the supplied reference crowns; absence of a polygon is **not** interpreted as absence of a tree.


In [ ]:
with rasterio.open(RASTER) as src:
    raster_crs=src.crs

g=gpd.read_file(ANNOT)
print("Raw Crown_Nadine polygons:",len(g))
print("Columns:",list(g.columns))
if g.crs is None:
    raise ValueError("Crown_Nadine.shp has no CRS; define it before benchmarking.")
if g.crs!=raster_crs:
    print("Reprojecting crowns:",g.crs,"->",raster_crs)
    g=g.to_crs(raster_crs)

g=g[g.geometry.notna() & ~g.geometry.is_empty].copy()
g=g[g.geometry.geom_type.isin(["Polygon","MultiPolygon"])].copy()
g["crown_area_m2"]=g.geometry.area.astype(float)
g=g[g.crown_area_m2>=MIN_AREA_M2].copy().reset_index(drop=True)
g["crown_id"]=np.arange(len(g),dtype=np.int64)
g["area_bin"]=pd.qcut(g.crown_area_m2,4,labels=False,duplicates="drop")

if N_CROWNS is None or len(g)<=N_CROWNS:
    sample=g.copy()
else:
    rng=np.random.default_rng(SEED)
    chosen=[]
    bins=sorted(g.area_bin.dropna().unique())
    for b in bins:
        a=g[g.area_bin==b]
        n=min(len(a),int(np.ceil(N_CROWNS/len(bins))))
        chosen.extend(rng.choice(a.index,size=n,replace=False).tolist())
    sample=g.loc[chosen].copy()
    if len(sample)>N_CROWNS:
        sample=sample.sample(N_CROWNS,random_state=SEED)
sample=sample.sort_values("crown_id").reset_index(drop=True)
sample["site"]=SITE

print("Eligible crowns >=",MIN_AREA_M2,"m²:",len(g))
print("Benchmark crowns:",len(sample))
display(sample.groupby("area_bin").size().rename("n").to_frame())
display(sample.crown_area_m2.describe(percentiles=[.05,.25,.5,.75,.95]).to_frame("crown_area_m2"))


In [ ]:
def prepare_chip(src,geom):
    w=rasterio.windows.from_bounds(*geom.bounds,transform=src.transform).round_offsets().round_lengths()
    bw,bh=max(1,int(w.width)),max(1,int(w.height))
    mx=max(int(bw*CROP_MARGIN_FRAC),(MIN_CROP_PX-bw)//2,32)
    my=max(int(bh*CROP_MARGIN_FRAC),(MIN_CROP_PX-bh)//2,32)
    ew=Window(int(w.col_off)-mx,int(w.row_off)-my,bw+2*mx,bh+2*my)
    rgb=np.moveaxis(src.read([1,2,3],window=ew,boundless=True,fill_value=0),0,-1).astype(np.uint8)
    tr=src.window_transform(ew)
    ref=rasterize([(geom,1)],out_shape=rgb.shape[:2],transform=tr,fill=0,dtype="uint8")
    rr,cc=np.where(ref>0)
    if len(rr)==0:
        raise ValueError("Reference crown rasterized to zero pixels")
    x0,x1=float(cc.min()),float(cc.max()+1); y0,y1=float(rr.min()),float(rr.max()+1)
    dx=(x1-x0)*BOX_EXPAND_FRAC; dy=(y1-y0)*BOX_EXPAND_FRAC
    box=[max(0,x0-dx),max(0,y0-dy),min(rgb.shape[1],x1+dx),min(rgb.shape[0],y1+dy)]
    return rgb,ref,box,abs(src.transform.a*src.transform.e)

manifest=[]
with rasterio.open(RASTER) as src:
    for row in tqdm(sample.itertuples(),total=len(sample),desc="PREP Baego"):
        stem=f"{SITE}_crown_{int(row.crown_id):05d}"
        rgbp=CHIPS/f"{stem}_rgb.png"; refp=CHIPS/f"{stem}_ref.png"
        rgb,ref,box,pixarea=prepare_chip(src,row.geometry)
        if FORCE_PREP or not (rgbp.exists() and refp.exists()):
            Image.fromarray(rgb).save(rgbp)
            Image.fromarray(ref*255).save(refp)
        manifest.append(dict(site=SITE,crown_id=int(row.crown_id),
            crown_area_m2=float(row.crown_area_m2),area_bin=int(row.area_bin),
            pixel_area_m2=float(pixarea),rgb_path=str(rgbp),ref_path=str(refp),
            box_x0=box[0],box_y0=box[1],box_x1=box[2],box_y1=box[3]))

man=pd.DataFrame(manifest)
MANIFEST=OUT/"benchmark_manifest.csv"
man.to_csv(MANIFEST,index=False)
print("Prepared",len(man),"Baego chips:",MANIFEST)


## 2 — Create an isolated official SAM3 environment

This is the key dependency fix. `/content/sam3env` is a separate virtual environment. Installing official SAM3 there may use NumPy 1.26.x without touching the host NumPy/Rasterio/GeoPandas installation.

The environment is ephemeral; the chips and results are persistent on Drive.


In [ ]:
# SAM3 needs numpy<2, while current Colab geospatial packages need numpy>=2.
# Use uv to create a real isolated Python 3.12 environment; do NOT modify host Python.
UV=Path("/content/uv")
SAMENV=Path("/content/sam3env")
SAMREPO=Path("/content/sam3")

if not UV.exists():
    subprocess.run(["bash","-lc","curl -LsSf https://astral.sh/uv/install.sh | env UV_INSTALL_DIR=/content sh"],check=True)
    # uv installer may name the binary directly under /content.
    candidates=[Path("/content/uv"),Path("/content/uv/uv"),Path.home()/".local/bin/uv"]
    UV=next((p for p in candidates if p.exists() and p.is_file()),None)
    assert UV is not None,"uv installation failed"
else:
    UV=Path("/content/uv")
print("uv:",subprocess.check_output([str(UV),"--version"],text=True).strip())

# Recreate only the isolated SAM environment if a previous failed venv is present.
if SAMENV.exists() and not (SAMENV/"bin/python").exists():
    shutil.rmtree(SAMENV)
if not (SAMENV/"bin/python").exists():
    subprocess.run([str(UV),"venv","--python","3.12",str(SAMENV)],check=True)

SAMPY=str(SAMENV/"bin/python")
# Reuse Colab's CUDA-enabled torch wheel via explicit install in the isolated env.
subprocess.run([str(UV),"pip","install","--python",SAMPY,
                "torch==2.10.0","torchvision",
                "--index-url","https://download.pytorch.org/whl/cu128"],check=True)

if not SAMREPO.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/facebookresearch/sam3.git",str(SAMREPO)],check=True)
else:
    subprocess.run(["git","-C",str(SAMREPO),"pull","--ff-only"],check=True)

# Official SAM3 currently requires numpy>=1.26,<2. Keep that constraint inside this env only.
subprocess.run([str(UV),"pip","install","--python",SAMPY,"setuptools<82"],check=True)
# The image builder imports modules that use packages listed by Meta under
# notebook/dev extras (not only the minimal core dependencies). Install the
# official extras rather than discovering them one-by-one at runtime.
subprocess.run([str(UV),"pip","install","--python",SAMPY,
                "-e",str(SAMREPO)+"[notebooks,dev]"],check=True)

subprocess.run([SAMPY,"-c",
    "import sys,numpy,torch,sam3; print('SAM Python',sys.version.split()[0],'numpy',numpy.__version__,'torch',torch.__version__,'CUDA',torch.cuda.is_available())"],
    check=True)
print("HOST unchanged: Python",sys.version.split()[0],"numpy",np.__version__,"rasterio",rasterio.__version__)


## 3 — Hugging Face authentication

SAM3 checkpoints are gated. First accept the official Meta SAM3 model terms in your Hugging Face account.

The recommended Colab method is a token stored in **Colab Secrets** as `HF_TOKEN` (key icon in the left sidebar). This avoids the short-lived device-code timeout. The token is passed only to the SAM3 subprocess and is never written to the notebook or Drive.

If `HF_TOKEN` is not present, this cell stops with instructions rather than waiting for an expiring device code.


In [ ]:
from google.colab import userdata
try:
    HF_TOKEN=userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN=None
assert HF_TOKEN, "Add your Hugging Face read token to Colab Secrets as HF_TOKEN, enable notebook access, then rerun this cell."
print("HF_TOKEN found in Colab Secrets (value not displayed).")


## 4 — Official SAM3 subprocess runner

The runner follows Meta's current image processor API directly:
- CUDA inference uses the same **bfloat16 autocast** setup as the official SAM3 examples;
- the manual crown bbox is converted from pixel `xyxy` to normalized `[center_x, center_y, width, height]`;
- the box is supplied through `Sam3Processor.add_geometric_prompt(..., label=True)`.

Completed masks are persistent and skipped unless `FORCE_SAM=True`.


In [ ]:
RUNNER=Path("/content/run_sam3_kvo05.py")
RUNNER.write_text(r'''import os,sys,json
from pathlib import Path
import numpy as np, pandas as pd, torch
from PIL import Image
from tqdm import tqdm
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

manifest=Path(sys.argv[1]); outdir=Path(sys.argv[2]); force=sys.argv[3]=="1"
outdir.mkdir(parents=True,exist_ok=True)
df=pd.read_csv(manifest)

if not torch.cuda.is_available():
    raise RuntimeError("KVO_05 requires a CUDA GPU runtime.")

# Match Meta's official SAM3 example inference setup.
torch.backends.cuda.matmul.allow_tf32=True
torch.backends.cudnn.allow_tf32=True
torch.autocast("cuda",dtype=torch.bfloat16).__enter__()
torch.inference_mode().__enter__()

print("Loading official SAM3...")
bpe_path="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
if not Path(bpe_path).exists():
    raise FileNotFoundError(f"SAM3 BPE vocabulary not found: {bpe_path}")
model=build_sam3_image_model(bpe_path=bpe_path)
processor=Sam3Processor(model)
print("SAM3 ready; CUDA:",torch.cuda.is_available())

def predict_box(img,box_xyxy):
    state=processor.set_image(img)
    W,H=img.size
    x0,y0,x1,y1=[float(v) for v in box_xyxy]
    # Official SAM3 geometric prompt format: normalized [cx,cy,w,h].
    box=[((x0+x1)/2)/W,((y0+y1)/2)/H,(x1-x0)/W,(y1-y0)/H]
    out=processor.add_geometric_prompt(box=box,label=True,state=state)
    masks=out["masks"]
    scores=out.get("scores",None)
    if torch.is_tensor(masks): masks=masks.detach().float().cpu().numpy()
    masks=np.asarray(masks)
    while masks.ndim>3 and masks.shape[0]==1: masks=masks[0]
    if masks.ndim==2: masks=masks[None]
    if masks.ndim==4 and masks.shape[1]==1: masks=masks[:,0]
    if len(masks)==0:
        return np.zeros((H,W),dtype=np.uint8),np.nan
    if scores is None:
        j=0; score=np.nan
    else:
        if torch.is_tensor(scores): scores=scores.detach().float().cpu().numpy()
        ss=np.asarray(scores).reshape(-1)
        j=int(ss.argmax()) if len(ss) else 0
        score=float(ss[j]) if len(ss) else np.nan
    return (masks[j]>0).astype(np.uint8),score

status=[]
for r in tqdm(df.itertuples(),total=len(df),desc="SAM3 Baego crowns"):
    stem=f"{r.site}_crown_{int(r.crown_id):05d}"
    mp=outdir/f"{stem}_sam3.png"; jp=outdir/f"{stem}_sam3.json"
    if mp.exists() and jp.exists() and not force:
        continue
    try:
        img=Image.open(r.rgb_path).convert("RGB")
        box=[r.box_x0,r.box_y0,r.box_x1,r.box_y1]
        mask,score=predict_box(img,box)
        if mask.shape!=(img.height,img.width):
            mask=np.asarray(Image.fromarray(mask*255).resize((img.width,img.height),Image.Resampling.NEAREST))>0
        Image.fromarray(mask.astype(np.uint8)*255).save(mp)
        jp.write_text(json.dumps({"score":score}))
        status.append({"site":r.site,"crown_id":int(r.crown_id),"ok":True,"error":""})
    except Exception as e:
        status.append({"site":r.site,"crown_id":int(r.crown_id),"ok":False,"error":repr(e)})
        pd.DataFrame(status).to_csv(outdir/"sam3_run_status.csv",index=False)
        raise
    if len(status)%10==0:
        pd.DataFrame(status).to_csv(outdir/"sam3_run_status.csv",index=False)
pd.DataFrame(status).to_csv(outdir/"sam3_run_status.csv",index=False)
print("DONE")
''')
print("Wrote:",RUNNER)


In [ ]:
env=os.environ.copy(); env["HF_TOKEN"]=HF_TOKEN
cmd=[SAMPY,"-u",str(RUNNER),str(MANIFEST),str(SAMOUT),"1" if FORCE_SAM else "0"]
print("Running:"," ".join(cmd))
p=subprocess.run(cmd,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print("\n========== SAM3 OUTPUT ==========\n")
print(p.stdout)
print("\n=================================")
print("Return code:",p.returncode)
if p.returncode!=0:
    raise RuntimeError("SAM3 subprocess failed; actual error is printed above.")


## 5 — Evaluate SAM3 against the high-quality Baego crowns

Primary metrics are object-wise IoU and Dice. Relative area bias tests whether SAM3 systematically over/undersegments crowns, and centroid displacement measures positional agreement.


In [ ]:
def mask_metrics(ref,pred,pixel_area):
    ref=ref>0; pred=pred>0
    inter=np.logical_and(ref,pred).sum(); union=np.logical_or(ref,pred).sum()
    nr=ref.sum(); npred=pred.sum()
    iou=inter/union if union else np.nan; dice=2*inter/(nr+npred) if nr+npred else np.nan
    ar=nr*pixel_area; ap=npred*pixel_area
    rb=np.argwhere(ref); pb=np.argwhere(pred); cd=np.nan
    if len(rb) and len(pb):
        d=rb.mean(0)-pb.mean(0); cd=float(np.sqrt((d*d).sum())*np.sqrt(pixel_area))
    return iou,dice,ar,ap,(ap-ar)/ar if ar else np.nan,cd

rows=[]
for r in tqdm(man.itertuples(),total=len(man),desc="EVALUATE"):
    stem=f"{r.site}_crown_{int(r.crown_id):05d}"; mp=SAMOUT/f"{stem}_sam3.png"; jp=SAMOUT/f"{stem}_sam3.json"
    if not mp.exists(): continue
    ref=np.asarray(Image.open(r.ref_path)); pred=np.asarray(Image.open(mp))
    iou,dice,ar,ap,bias,cd=mask_metrics(ref,pred,float(r.pixel_area_m2))
    score=json.loads(jp.read_text()).get("score",np.nan) if jp.exists() else np.nan
    rows.append(dict(site=r.site,crown_id=int(r.crown_id),class_id=int(r.class_id),truth=int(r.truth),
        crown_area_m2=float(r.crown_area_m2),area_bin=int(r.area_bin),sam_score=score,iou=iou,dice=dice,
        reference_area_m2=ar,sam_area_m2=ap,relative_area_bias=bias,centroid_displacement_m=cd))
res=pd.DataFrame(rows); RESULTS=OUT/"sam3_crown_metrics.csv"; res.to_csv(RESULTS,index=False)
print("Evaluated",len(res),"of",len(man),"crowns")
summary=res.groupby("site").agg(n=("iou","size"),median_iou=("iou","median"),mean_iou=("iou","mean"),
    median_dice=("dice","median"),mean_dice=("dice","mean"),
    median_abs_area_bias=("relative_area_bias",lambda x: np.median(np.abs(x))),
    median_centroid_displacement_m=("centroid_displacement_m","median")).reset_index()
display(summary); summary.to_csv(OUT/"sam3_site_summary.csv",index=False)


In [ ]:
fig,ax=plt.subplots(figsize=(7,5))
for site,d in res.groupby("site"): ax.scatter(d.crown_area_m2,d.iou,s=15,alpha=.45,label=site)
ax.set_xscale("log"); ax.set_ylim(-.02,1.02); ax.set_xlabel("Reference crown area (m², log scale)"); ax.set_ylabel("SAM3 IoU")
ax.set_title("SAM3 crown delineation vs crown size"); ax.legend(); fig.tight_layout()
fig.savefig(OUT/"sam3_iou_vs_crown_area.png",dpi=180); plt.show()

fig,ax=plt.subplots(figsize=(7,5))
ax.boxplot([res.loc[res.site==s,"iou"].dropna().values for s in SITES],tick_labels=list(SITES),showfliers=False)
ax.set_ylim(0,1); ax.set_ylabel("IoU"); ax.set_title("SAM3 crown IoU by site"); fig.tight_layout()
fig.savefig(OUT/"sam3_iou_by_site.png",dpi=180); plt.show()


## 6 — Visual QA: best, median and worst Baego crowns

Solid contour = high-quality Crown_Nadine reference; dashed contour = SAM3. The expanded prompt box is shown.


In [ ]:
def draw_saved_case(r,label):
    rgb=np.asarray(Image.open(r.rgb_path)); ref=np.asarray(Image.open(r.ref_path))>0
    stem=f"{r.site}_crown_{int(r.crown_id):05d}"; pred=np.asarray(Image.open(SAMOUT/f"{stem}_sam3.png"))>0
    fig,ax=plt.subplots(figsize=(6,6)); ax.imshow(rgb)
    ax.contour(ref.astype(float),levels=[.5],linewidths=2); ax.contour(pred.astype(float),levels=[.5],linewidths=2,linestyles="--")
    ax.add_patch(plt.Rectangle((r.box_x0,r.box_y0),r.box_x1-r.box_x0,r.box_y1-r.box_y0,fill=False,linewidth=1.2))
    m=res[(res.site==r.site)&(res.crown_id==r.crown_id)].iloc[0]
    ax.set_title(f"{r.site} — {label} — IoU={m.iou:.3f}, Dice={m.dice:.3f}"); ax.axis("off"); fig.tight_layout()
    fig.savefig(VIS/f"{r.site}_{label}_crown_{int(r.crown_id)}.png",dpi=180); plt.show()

for site,d in res.groupby("site"):
    d=d.sort_values("iou"); ids={"worst":int(d.iloc[0].crown_id),"median":int(d.iloc[len(d)//2].crown_id),"best":int(d.iloc[-1].crown_id)}
    for label,cid in ids.items():
        r=man[(man.site==site)&(man.crown_id==cid)].iloc[0]
        draw_saved_case(r,label)


## 7 — Decision gate for NB06

This Baego benchmark evaluates **prompt-conditioned crown delineation**, not automatic crown discovery. Because Crown_Nadine is not wall-to-wall, do not use unannotated areas as negatives and do not calculate regional detection completeness from this notebook.

If zero-shot SAM3 reproduces these high-quality crowns well, NB06 can test automatic/exemplar crown discovery and then combine SAM-derived crown objects with the KVO_04 Musanga probability surface. If performance is weak, these detailed crowns provide a clean reference for diagnosing prompt and boundary errors.
